# Generate Pipeline (Unified)

`vocab_alignment.ipynb`, `generate/recipe_molecule_pipeline_redesign.ipynb`, `analysis/cuisine_only_cluster_analysis.ipynb`의 핵심 로직을 하나로 정리한 노트북입니다.

## 목표
- 입력: `../recipes.csv`, `../preprocess/flavordb_alias_in_vocab_only.csv`, `../molecules.csv`
- 처리: FlavorDB2 재료를 **하나 이상 포함하는 레시피만** 대상으로 분석
- 출력:
  - cuisine별 + ALL 레벨 graph csv 저장
  - cuisine별 + ALL 분자 그래프 클러스터링
  - cluster lift 결과(csv) 및 시각화(plot)
  - cluster graph 시각화(plot)

## 핵심 수식
\[
S(r,m)=\sum_{i \in I(r)} w_{r,i}\cdotrac{\mathbf{1}\{m \in M(i)\}}{|M(i)|}
\]

## 0) 환경 설정
- 긴 루프에는 `tqdm`을 사용합니다.
- 중간 산출물은 `shape`, `head(10)`을 출력합니다.

In [ ]:
from __future__ import annotations

import ast
import re
import time
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(x, **kwargs):
        return x

plt.style.use('seaborn-v0_8-whitegrid')
START_TS = time.time()

In [ ]:
# Paths
RECIPES_PATH = Path('../recipes.csv')
FLAVORDB_PATH = Path('../preprocess/flavordb_alias_in_vocab_only.csv')
MOLECULES_PATH = Path('../molecules.csv')

OUT_DIR = Path('../result')
GRAPH_DIR = OUT_DIR / 'graph'
ANALYSIS_DIR = OUT_DIR / 'analysis'
PLOT_DIR = OUT_DIR / 'plots'

for d in [OUT_DIR, GRAPH_DIR, ANALYSIS_DIR, PLOT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('Output directories:')
for d in [GRAPH_DIR, ANALYSIS_DIR, PLOT_DIR]:
    print(' -', d.resolve())

## 1) 데이터 로드 + 유틸 함수

In [ ]:
def norm_text(s: str) -> str:
    if pd.isna(s):
        return ''
    s = str(s).strip().lower()
    s = re.sub(r'[_\-]+', ' ', s)
    s = re.sub(r'\s+', ' ', s)
    s = re.sub(r"[^\w\s']", '', s)
    return s.strip()


def find_col(df: pd.DataFrame, candidates: list[str]) -> str | None:
    lower_map = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    return None


def parse_ingredient_ratio(raw):
    if pd.isna(raw):
        return []

    if isinstance(raw, str):
        txt = raw.strip()
        if not txt:
            return []
        try:
            raw = ast.literal_eval(txt)
        except Exception:
            return []

    out = []
    if isinstance(raw, dict):
        for k, v in raw.items():
            try:
                out.append((norm_text(k), float(v)))
            except Exception:
                continue
    elif isinstance(raw, list):
        for item in raw:
            if isinstance(item, dict):
                ing = item.get('ingredient', item.get('name', item.get('ing')))
                val = item.get('ratio', item.get('weight', item.get('w', item.get('gram', 0))))
                try:
                    out.append((norm_text(ing), float(val)))
                except Exception:
                    continue
            elif isinstance(item, (list, tuple)) and len(item) >= 2:
                try:
                    out.append((norm_text(item[0]), float(item[1])))
                except Exception:
                    continue

    return [(i, w) for i, w in out if i and np.isfinite(w) and w > 0]

In [ ]:
recipes = pd.read_csv(RECIPES_PATH)
flavordb = pd.read_csv(FLAVORDB_PATH)
molecules = pd.read_csv(MOLECULES_PATH)

print('recipes:', recipes.shape)
print('flavordb:', flavordb.shape)
print('molecules:', molecules.shape)

display(recipes.head(10))
display(flavordb.head(10))
display(molecules.head(10))

In [ ]:
recipe_id_col = find_col(recipes, ['recipe_id', 'id', 'rid'])
cuisine_col = find_col(recipes, ['cuisine', 'cuisine_name'])
ratio_col = find_col(recipes, ['ingredients_ratio', 'ingredient_ratio', 'ingredients_with_ratio'])
if ratio_col is None:
    ratio_col = find_col(recipes, ['cleaned_ingredients', 'ingredients'])

if cuisine_col is None or ratio_col is None:
    raise KeyError(f'Required columns missing. cuisine_col={cuisine_col}, ratio_col={ratio_col}, all={list(recipes.columns)}')

if recipe_id_col is None:
    recipes = recipes.copy()
    recipes['recipe_id'] = np.arange(len(recipes), dtype=int)
    recipe_id_col = 'recipe_id'

ing_col = find_col(flavordb, ['ingredient', 'entity_alias_readable', 'entity_alias', 'name'])
mol_col = find_col(flavordb, ['molecule id', 'molecule_id', 'pubchem id', 'pubchem_id'])

if ing_col is None or mol_col is None:
    raise KeyError(f'FlavorDB columns missing. ing_col={ing_col}, mol_col={mol_col}, all={list(flavordb.columns)}')

print('recipe_id_col:', recipe_id_col)
print('cuisine_col:', cuisine_col)
print('ratio_col:', ratio_col)
print('flavor ingredient col:', ing_col)
print('flavor molecule col:', mol_col)

## 2) FlavorDB ingredient → molecule set $M(i)$ 생성

In [ ]:
flavordb = flavordb.copy()
flavordb['ingredient_norm'] = flavordb[ing_col].map(norm_text)
flavordb['molecule_id'] = pd.to_numeric(flavordb[mol_col], errors='coerce')

ing_to_molecules = (
    flavordb.dropna(subset=['ingredient_norm', 'molecule_id'])
    .groupby('ingredient_norm')['molecule_id']
    .apply(lambda x: sorted(set(int(v) for v in x if pd.notna(v))))
    .to_dict()
)

print('mapped ingredients:', len(ing_to_molecules))
print('sample mapping:', list(ing_to_molecules.items())[:5])

## 3) 레시피 long format 생성 + FlavorDB2 포함 레시피 필터링

In [ ]:
records = []
for _, row in tqdm(recipes.iterrows(), total=len(recipes), desc='Parse recipes'):
    rid = row[recipe_id_col]
    cuisine = row[cuisine_col]
    pairs = parse_ingredient_ratio(row[ratio_col])

    z = sum(w for _, w in pairs)
    if z <= 0:
        continue

    for ing, w in pairs:
        records.append({
            'recipe_id': int(rid),
            'cuisine': str(cuisine),
            'ingredient': ing,
            'w_ri': float(w / z),
        })

recipes_long = pd.DataFrame(records)
print('recipes_long shape:', recipes_long.shape)
display(recipes_long.head(10))

if recipes_long.empty:
    raise RuntimeError('No parsed recipe rows. Check ratio_col parsing logic and input file format.')

recipes_long['has_flavordb_match'] = recipes_long['ingredient'].isin(ing_to_molecules)
matched_rids = set(recipes_long.loc[recipes_long['has_flavordb_match'], 'recipe_id'].unique())
recipes_long = recipes_long[recipes_long['recipe_id'].isin(matched_rids)].copy()

print('filtered recipes_long shape:', recipes_long.shape)
print('n_recipes_with_flavordb_ingredient:', recipes_long['recipe_id'].nunique())
display(recipes_long.head(10))

recipes_long.to_csv(OUT_DIR / 'recipes_long_normalized.csv', index=False)

## 4) 수식 기반 recipe-molecule score 계산

아래 수식으로 $S(r,m)$을 계산합니다.
\[
S(r,m)=\sum_{i \in I(r)} w_{r,i}\cdotrac{\mathbf{1}\{m \in M(i)\}}{|M(i)|}
\]

In [ ]:
score_rows = []
unk_rows = []

for rid, grp in tqdm(recipes_long.groupby('recipe_id'), desc='Compute S(r,m)'):
    acc = defaultdict(float)
    mass_covered = 0.0

    for ing, w in grp[['ingredient', 'w_ri']].itertuples(index=False):
        mols = ing_to_molecules.get(ing, [])
        if not mols:
            continue
        contrib = w / len(mols)
        for m in mols:
            acc[m] += contrib
        mass_covered += w

    for m, s in acc.items():
        score_rows.append({'recipe_id': int(rid), 'molecule_id': int(m), 'S_rm': float(s)})

    unk_rows.append({'recipe_id': int(rid), 'u_r': float(max(0.0, 1.0 - mass_covered))})

recipe_molecule = pd.DataFrame(score_rows)
recipe_unk = pd.DataFrame(unk_rows)

rid_to_cuisine = (
    recipes_long[['recipe_id', 'cuisine']]
    .drop_duplicates()
    .set_index('recipe_id')['cuisine']
    .to_dict()
)
recipe_molecule['cuisine'] = recipe_molecule['recipe_id'].map(rid_to_cuisine)

print('recipe_molecule:', recipe_molecule.shape)
print('recipe_unk:', recipe_unk.shape)
display(recipe_molecule.head(10))
display(recipe_unk.head(10))

recipe_molecule.to_csv(GRAPH_DIR / 'recipe_molecule_edges.csv', index=False)
recipe_unk.to_csv(GRAPH_DIR / 'recipe_unk_mass.csv', index=False)

## 5) Cuisine별 + ALL graph csv 저장

In [ ]:
def safe_key(name: str) -> str:
    return re.sub(r'\W+', '_', str(name)).strip('_') or 'unknown'


def export_graph_for_group(df_edges: pd.DataFrame, key: str):
    out_edges = GRAPH_DIR / f'{key}_recipe_molecule_edges.csv'
    out_nodes = GRAPH_DIR / f'{key}_molecule_nodes.csv'

    mol_weight = (
        df_edges.groupby('molecule_id', as_index=False)['S_rm']
        .sum()
        .rename(columns={'S_rm': 'weight'})
        .sort_values('weight', ascending=False)
    )

    df_edges.to_csv(out_edges, index=False)
    mol_weight.to_csv(out_nodes, index=False)

    print(f'[{key}] edges={df_edges.shape}, nodes={mol_weight.shape}')
    display(df_edges.head(10))
    display(mol_weight.head(10))


for cuisine, sub in tqdm(recipe_molecule.groupby('cuisine'), desc='Export cuisine graphs'):
    export_graph_for_group(sub.copy(), safe_key(cuisine))

export_graph_for_group(recipe_molecule.copy(), 'ALL')

## 6) Molecule graph 구성 + clustering + ingredient lift + 시각화

In [ ]:
def build_recipe_to_molset(df_edges: pd.DataFrame):
    d = defaultdict(set)
    for rid, mid in df_edges[['recipe_id', 'molecule_id']].itertuples(index=False):
        d[int(rid)].add(int(mid))
    return d


def build_molecule_graph(rids, rid2molset, min_cooc=3):
    cooc = Counter()
    for rid in rids:
        mols = sorted(rid2molset.get(int(rid), []))
        for i in range(len(mols)):
            for j in range(i + 1, len(mols)):
                a, b = mols[i], mols[j]
                cooc[(a, b)] += 1

    G = nx.Graph()
    for (a, b), w in cooc.items():
        if w >= min_cooc:
            G.add_edge(a, b, weight=w)
    return G


def cluster_graph(G: nx.Graph):
    if G.number_of_nodes() == 0:
        return {}
    communities = nx.algorithms.community.louvain_communities(G, weight='weight', seed=42)
    part = {}
    for cid, comm in enumerate(communities):
        for node in comm:
            part[node] = cid
    return part


def ingredient_lift(recipes_subset: pd.DataFrame, ingredient_col='ingredient', target_flag_col='in_cluster'):
    base = recipes_subset.groupby(ingredient_col).size().rename('base_n')
    sub = recipes_subset[recipes_subset[target_flag_col] == 1].groupby(ingredient_col).size().rename('cluster_n')
    m = pd.concat([base, sub], axis=1).fillna(0)

    N = len(recipes_subset)
    K = max(1, int((recipes_subset[target_flag_col] == 1).sum()))
    m['p_base'] = m['base_n'] / max(1, N)
    m['p_cluster'] = m['cluster_n'] / K
    m['lift'] = m['p_cluster'] / m['p_base'].replace(0, np.nan)
    return m.sort_values('lift', ascending=False)

In [ ]:
rid2molset = build_recipe_to_molset(recipe_molecule)
all_cuisines = sorted(recipes_long['cuisine'].dropna().unique().tolist())
target_groups = ['ALL'] + all_cuisines

analysis_summary = []

for group in tqdm(target_groups, desc='Cuisine clustering'):
    if group == 'ALL':
        rids = sorted(recipes_long['recipe_id'].unique())
    else:
        rids = sorted(recipes_long.loc[recipes_long['cuisine'] == group, 'recipe_id'].unique())

    print(f'\n==== {group} ====')
    print('recipes:', len(rids))

    G = build_molecule_graph(rids, rid2molset, min_cooc=3)
    part = cluster_graph(G)

    n_clusters = len(set(part.values())) if part else 0
    print('graph nodes:', G.number_of_nodes(), 'edges:', G.number_of_edges(), 'clusters:', n_clusters)

    fig = plt.figure(figsize=(8, 6))
    if G.number_of_nodes() > 0:
        pos = nx.spring_layout(G, seed=42)
        node_colors = [part.get(n, -1) for n in G.nodes()]
        nx.draw_networkx(
            G,
            pos=pos,
            node_size=40,
            with_labels=False,
            node_color=node_colors,
            cmap='tab20',
            edge_color='gray',
            alpha=0.8,
        )
        plt.title(f'{group} molecule cluster graph')
    else:
        plt.text(0.5, 0.5, 'Empty graph', ha='center', va='center')
        plt.title(f'{group} molecule cluster graph')
    fig.tight_layout()
    fig.savefig(PLOT_DIR / f'{safe_key(group)}_cluster_graph.png', dpi=150)
    plt.close(fig)

    rid_rows = []
    for rid in rids:
        cnt = Counter(part.get(m) for m in rid2molset.get(rid, set()) if m in part)
        cid = cnt.most_common(1)[0][0] if cnt else -1
        rid_rows.append((rid, cid))
    rid_cluster = pd.DataFrame(rid_rows, columns=['recipe_id', 'cluster_id'])

    tmp = recipes_long[recipes_long['recipe_id'].isin(rids)][['recipe_id', 'ingredient']].copy()
    tmp = tmp.merge(rid_cluster, on='recipe_id', how='left')

    lift_records = []
    valid_clusters = sorted(c for c in rid_cluster['cluster_id'].unique() if c >= 0)
    for cid in valid_clusters:
        tmp['in_cluster'] = (tmp['cluster_id'] == cid).astype(int)
        lf = ingredient_lift(tmp, ingredient_col='ingredient', target_flag_col='in_cluster').head(20).reset_index()
        lf.insert(0, 'group', group)
        lf.insert(1, 'cluster_id', cid)
        lift_records.append(lf)

    if lift_records:
        lift_df = pd.concat(lift_records, ignore_index=True)
    else:
        lift_df = pd.DataFrame(columns=['group', 'cluster_id', 'ingredient', 'base_n', 'cluster_n', 'p_base', 'p_cluster', 'lift'])

    lift_path = ANALYSIS_DIR / f'{safe_key(group)}_cluster_lift_top20.csv'
    lift_df.to_csv(lift_path, index=False)

    fig = plt.figure(figsize=(7, 4))
    dist = rid_cluster['cluster_id'].value_counts().sort_index()
    dist = dist[dist.index >= 0]
    if len(dist):
        dist.plot(kind='bar')
        plt.ylabel('# recipes')
        plt.title(f'{group} cluster distribution')
    else:
        plt.text(0.5, 0.5, 'No cluster assignment', ha='center', va='center')
        plt.title(f'{group} cluster distribution')
    fig.tight_layout()
    fig.savefig(PLOT_DIR / f'{safe_key(group)}_cluster_distribution.png', dpi=150)
    plt.close(fig)

    print('lift rows:', len(lift_df))
    display(lift_df.head(10))

    analysis_summary.append({
        'group': group,
        'n_recipes': len(rids),
        'n_graph_nodes': G.number_of_nodes(),
        'n_graph_edges': G.number_of_edges(),
        'n_clusters': n_clusters,
        'lift_rows': len(lift_df),
    })

## 7) 요약 저장

In [ ]:
summary_df = pd.DataFrame(analysis_summary)
summary_df.to_csv(ANALYSIS_DIR / 'analysis_summary.csv', index=False)

print('analysis summary:', summary_df.shape)
display(summary_df.head(10))

elapsed = time.time() - START_TS
print(f'Done. elapsed={elapsed:.1f}s')
print('Check outputs in:')
print(' -', GRAPH_DIR.resolve())
print(' -', ANALYSIS_DIR.resolve())
print(' -', PLOT_DIR.resolve())